# Risk Measures Analysis — XTB S.A.
## Enterprise Risk Management — Project 3

**Risk variables:**
- **EUR/PLN** — euro to zloty exchange rate
- **USD/PLN** — US dollar to zloty exchange rate  
- **GBP/PLN** — British pound to zloty exchange rate
- **XTB.WA** — XTB share price on the Warsaw Stock Exchange (GPW)

XTB S.A. is a forex/CFD broker listed on the Warsaw Stock Exchange (GPW). The company's revenue depends strongly on exchange rates (commissions from client transactions) and on its stock market valuation. Approximately 50% of revenue is generated in EUR, 30% in USD, and 20% in GBP — we will use these weights to build a currency portfolio.

**Approach:** parametric (fitting Normal and t-Student distributions) + non-parametric (empirical measures) — comparison of both.

**Data:** daily closing prices over the last 8 years (via `yfinance`). As the risk variable we use **logarithmic returns** (log-returns), which eliminates non-stationarity and autocorrelation in price levels.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.stats import norm, t as t_dist, genextreme, genpareto
from scipy.stats import kstest, anderson
import warnings
warnings.filterwarnings('ignore')
warnings.catch_warnings()

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

# ---------- Data download ----------
tickers = {
    'EURPLN=X': 'EUR/PLN',
    'USDPLN=X': 'USD/PLN',
    'GBPPLN=X': 'GBP/PLN',
    'XTB.WA':   'XTB.WA'
}

data = yf.download(list(tickers.keys()), start='2018-01-01', end='2025-12-31')['Close']
data.columns = [tickers.get(c, c) for c in data.columns]
data = data.dropna()

# Logarithmic returns
log_returns = np.log(data / data.shift(1)).dropna()

print(f"Period: {data.index[0].date()} — {data.index[-1].date()}")
print(f"Number of observations (prices): {len(data)}")
print(f"Number of observations (log-returns): {len(log_returns)}")
print(f"\nDescriptive statistics of log-returns (daily):")
log_returns.describe().T

In [ ]:
# ---------- Stationarity test + ACF / PACF ----------
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf


# === ADF and KPSS tests for closing prices and log-returns ===
print("=" * 70)
print("STATIONARITY TEST — ADF & KPSS")
print("=" * 70)

for label, series_dict in [("Closing prices", data), ("Log-returns", log_returns)]:
    print(f"\n{'─'*70}")
    print(f"  {label}")
    print(f"{'─'*70}")
    rows = []
    for col in log_returns.columns:
        s = series_dict[col].dropna()
        # ADF (H0: unit root present → non-stationary)
        adf_stat, adf_p, adf_lags, *_ = adfuller(s, autolag='AIC')
        # KPSS (H0: series is stationary)
        kpss_stat, kpss_p, kpss_lags, kpss_crit = kpss(s, regression='c', nlags='auto')
        rows.append({
            'Variable': col,
            'ADF stat': round(adf_stat, 4),
            'ADF p-value': round(adf_p, 4),
            'ADF lags': adf_lags,
            'KPSS stat': round(kpss_stat, 4),
            'KPSS p-value': round(kpss_p, 4),
            'Stationary?': 'YES' if adf_p < 0.05 and kpss_p > 0.05 else 'NO'
        })
    df_tests = pd.DataFrame(rows).set_index('Variable')
    print(df_tests.to_string())
"""
# === ACF and PACF for log-returns ===
fig, axes = plt.subplots(len(log_returns.columns), 2, figsize=(16, 3.5 * len(log_returns.columns)))
fig.suptitle('ACF and PACF — logarithmic returns', fontsize=14, fontweight='bold')

for i, col in enumerate(log_returns.columns):
    plot_acf(log_returns[col].dropna(), ax=axes[i, 0], lags=40, title=f'ACF — {col}', alpha=0.05)
    plot_pacf(log_returns[col].dropna(), ax=axes[i, 1], lags=40, title=f'PACF — {col}', alpha=0.05, method='ywm')
    axes[i, 0].grid(alpha=0.3)
    axes[i, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# === ACF and PACF for squared log-returns (ARCH effect) ===
fig, axes = plt.subplots(len(log_returns.columns), 2, figsize=(16, 3.5 * len(log_returns.columns)))
fig.suptitle('ACF and PACF — squared log-returns (volatility clustering effect)', fontsize=14, fontweight='bold')

for i, col in enumerate(log_returns.columns):
    sq = log_returns[col].dropna() ** 2
    plot_acf(sq, ax=axes[i, 0], lags=40, title=f'ACF r² — {col}', alpha=0.05)
    plot_pacf(sq, ax=axes[i, 1], lags=40, title=f'PACF r² — {col}', alpha=0.05, method='ywm')
    axes[i, 0].grid(alpha=0.3)
    axes[i, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
"""

In [ ]:
# ---------- Data visualization ----------
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Closing prices — XTB S.A. risk variables', fontsize=14, fontweight='bold')

for ax, col in zip(axes.flat, log_returns.columns):
    data[col].plot(ax=ax, color='steelblue', linewidth=0.8)
    ax.set_title(col)
    ax.set_ylabel('Rate / Price')
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Logarithmic returns (daily)', fontsize=14, fontweight='bold')

for ax, col in zip(axes.flat, log_returns.columns):
    log_returns[col].plot(ax=ax, color='darkred', linewidth=0.5, alpha=0.7)
    ax.set_title(col)
    ax.set_ylabel('Log-return')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
# 1. Univariate case

For each risk variable separately we compute:
- **a) Volatility measures:** standard deviation, variance, interquartile range (IQR), MAD
- **b) Quantiles:** VaR at 95%, 99% levels (empirical and parametric)
- **c) Distribution function values:** F(0), F(μ±σ), F(μ±2σ) + graphical comparison of empirical vs theoretical CDF

In [ ]:
# ============================================================
# 1a. Volatility measures — univariate case
# ============================================================

def compute_volatility_measures(returns_series):
    """Computes volatility measures for a single variable."""
    return {
        'Mean (μ)': returns_series.mean(),
        'Std. dev. (σ)': returns_series.std(),
        'Variance (σ²)': returns_series.var(),
        'σ annualized': returns_series.std() * np.sqrt(252),
        'IQR': returns_series.quantile(0.75) - returns_series.quantile(0.25),
        'MAD': np.median(np.abs(returns_series - returns_series.median())),
        'Skewness': returns_series.skew(),
        'Kurtosis (exc.)': returns_series.kurtosis(),
    }

vol_table = pd.DataFrame({col: compute_volatility_measures(log_returns[col]) 
                           for col in log_returns.columns})

print("=" * 70)
print("1a. VOLATILITY MEASURES — univariate case")
print("=" * 70)
vol_table.round(6)

In [ ]:
# ============================================================
# 1b. Quantiles — empirical and parametric VaR (Value at Risk)
# ============================================================

alpha_levels = [0.01, 0.05, 0.10]

print("=" * 70)
print("1b. QUANTILES / VaR — univariate case")
print("=" * 70)

# Distribution fitting: Normal and t-Student
fitted_params = {}

for col in log_returns.columns:
    r = log_returns[col].values
    mu, sigma = r.mean(), r.std()
    # t-Student: MLE
    df_t, loc_t, scale_t = t_dist.fit(r)
    fitted_params[col] = {
        'norm': (mu, sigma),
        't': (df_t, loc_t, scale_t)
    }

# VaR table
var_results = []
for col in log_returns.columns:
    r = log_returns[col].values
    mu_n, sig_n = fitted_params[col]['norm']
    df_t, loc_t, scale_t = fitted_params[col]['t']
    
    for alpha in alpha_levels:
        var_emp = np.percentile(r, alpha * 100)
        var_norm = norm.ppf(alpha, loc=mu_n, scale=sig_n)
        var_t = t_dist.ppf(alpha, df_t, loc=loc_t, scale=scale_t)
        
        # ES (Expected Shortfall / CVaR) — average losses below VaR
        es_emp = r[r <= var_emp].mean()
        
        var_results.append({
            'Variable': col,
            'α': f'{alpha:.0%}',
            'VaR emp.': f'{var_emp:.5f}',
            'VaR Normal': f'{var_norm:.5f}',
            'VaR t-Student': f'{var_t:.5f}',
            'ES emp.': f'{es_emp:.5f}',
        })

var_df = pd.DataFrame(var_results)
print("\nVaR = α quantile (interpretation: with probability α the loss will exceed |VaR|)")
print("ES = Expected Shortfall (average losses in the tail below VaR)\n")
var_df

In [ ]:
# ============================================================
# 1c. Distribution functions — empirical vs theoretical comparison
# ============================================================

print("=" * 70)
print("1c. DISTRIBUTION FUNCTIONS — univariate case")
print("=" * 70)

# Distribution function values at selected points
print("\nEmpirical and theoretical distribution function values at selected points:")
print("(F(0) = P(R ≤ 0), i.e. probability of loss)\n")

cdf_results = []
for col in log_returns.columns:
    r = log_returns[col].values
    mu_n, sig_n = fitted_params[col]['norm']
    df_t, loc_t, scale_t = fitted_params[col]['t']
    
    points = {
        'F(0)': 0,
        'F(μ-2σ)': mu_n - 2*sig_n,
        'F(μ-σ)': mu_n - sig_n,
        'F(μ)': mu_n,
        'F(μ+σ)': mu_n + sig_n,
        'F(μ+2σ)': mu_n + 2*sig_n,
    }
    
    for label, x in points.items():
        f_emp = np.mean(r <= x)
        f_norm = norm.cdf(x, loc=mu_n, scale=sig_n)
        f_t = t_dist.cdf(x, df_t, loc=loc_t, scale=scale_t)
        cdf_results.append({
            'Variable': col, 'Point': label, 'x': f'{x:.6f}',
            'F_emp': f'{f_emp:.4f}', 'F_norm': f'{f_norm:.4f}', 'F_t': f'{f_t:.4f}'
        })

cdf_df = pd.DataFrame(cdf_results)
cdf_df

In [ ]:
# Graphical comparison of histograms with fitted densities
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Distribution comparison: empirical vs Normal vs t-Student', fontsize=14, fontweight='bold')

for i, col in enumerate(log_returns.columns):
    r = log_returns[col].values
    mu_n, sig_n = fitted_params[col]['norm']
    df_t, loc_t, scale_t = fitted_params[col]['t']
    x_grid = np.linspace(r.min(), r.max(), 500)

    ax = axes[i]
    ax.hist(r, bins=80, density=True, alpha=0.5, color='steelblue', label='Empirical')
    ax.plot(x_grid, norm.pdf(x_grid, mu_n, sig_n), 'r-', lw=2, label='Normal')
    ax.plot(x_grid, t_dist.pdf(x_grid, df_t, loc_t, scale_t), 'g--', lw=2, label=f't (df={df_t:.1f})')
    ax.set_title(f'{col} — density')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Distribution goodness-of-fit test — Kolmogorov-Smirnov
print("Kolmogorov-Smirnov test (H0: data come from the given distribution):\n")

ks_results = []
for col in log_returns.columns:
    r = log_returns[col].values
    mu_n, sig_n = fitted_params[col]['norm']
    df_t, loc_t, scale_t = fitted_params[col]['t']
    
    ks_norm = kstest(r, 'norm', args=(mu_n, sig_n))
    ks_t = kstest(r, 't', args=(df_t, loc_t, scale_t))
    
    ks_results.append({
        'Variable': col,
        'KS stat (Normal)': f'{ks_norm.statistic:.4f}',
        'p-value (Normal)': f'{ks_norm.pvalue:.4f}',
        'KS stat (t-Stud.)': f'{ks_t.statistic:.4f}',
        'p-value (t-Stud.)': f'{ks_t.pvalue:.4f}',
    })

ks_df = pd.DataFrame(ks_results)
print("p-value < 0.05 → reject H0 (distribution does not fit)")
print("p-value ≥ 0.05 → no grounds to reject H0\n")
ks_df

In [ ]:
# ============================================================
# 1b+. Fitting alternative distributions for USD/PLN and XTB.WA
# (Normal and t-Student distributions rejected by KS test)
# ============================================================
from scipy.stats import laplace, skewnorm, gennorm, johnsonsu, nct

candidate_dists = {
    'Laplace':           laplace,
    'Skew-Normal':       skewnorm,
    'Gen. Normal (GGD)': gennorm,      # β<2 → heavier tails than Normal
    'Johnson SU':        johnsonsu,
    'NoncentralT':       nct,
}

rejected_cols = ['USD/PLN', 'XTB.WA']

print('=' * 80)
print('FITTING ALTERNATIVE DISTRIBUTIONS (USD/PLN, XTB.WA)')
print('KS test rejected Normal and t-Student distributions for these variables.')
print('=' * 80)

alt_fit_results = []

for col in rejected_cols:
    r = log_returns[col].values
    print(f"\n{'─'*60}")
    print(f'  {col}')
    print(f"{'─'*60}")
    
    best_pval = -1
    best_name = None
    
    for dist_name, dist_obj in candidate_dists.items():
        try:
            params = dist_obj.fit(r)
            ks_stat, ks_pval = kstest(r, dist_obj.cdf, args=params)
            
            alt_fit_results.append({
                'Variable': col,
                'Distribution': dist_name,
                'KS stat': f'{ks_stat:.4f}',
                'p-value': f'{ks_pval:.4f}',
                'Conclusion': '✓ do not reject H0' if ks_pval >= 0.05 else '✗ reject H0',
                'Parameters': ', '.join(f'{p:.4f}' for p in params),
            })
            
            if ks_pval > best_pval:
                best_pval = ks_pval
                best_name = dist_name
                
            print(f'  {dist_name:20s}  KS={ks_stat:.4f}  p={ks_pval:.4f}'
                  f"  {'✓' if ks_pval >= 0.05 else '✗'}")
        except Exception as e:
            print(f'  {dist_name:20s}  ERROR: {e}')
    
    print(f'\n  >>> Best fit: {best_name} (p-value = {best_pval:.4f})')

alt_fit_df = pd.DataFrame(alt_fit_results)
print('\n')
display(alt_fit_df)

# ============================================================
# Additional test: alpha-stable (Lévy alpha-stable) distribution for XTB.WA
# ============================================================
from scipy.stats import levy_stable

print('\n' + '=' * 80)
print('ALPHA-STABLE DISTRIBUTION (Lévy stable) — XTB.WA')
print('=' * 80)

r_xtb = log_returns['XTB.WA'].values
levy_stable.fit_method = 'quantile'    # McCulloch method — fast
levy_stable.pdf_default_method = 'fft-simpson'
alpha_params = levy_stable.fit(r_xtb)
ks_stat, ks_pval = kstest(r_xtb, levy_stable.cdf, args=alpha_params)

print(f'  Parameters: α={alpha_params[0]:.4f}, β={alpha_params[1]:.4f}, '
      f'loc={alpha_params[2]:.6f}, scale={alpha_params[3]:.6f}')
print(f'  KS stat = {ks_stat:.4f},  p-value = {ks_pval:.4f}')
print(f"  Conclusion: {'✓ do not reject H0 — alpha-stable distribution fits' if ks_pval >= 0.05 else '✗ reject H0 — alpha-stable distribution does not fit'}")


---
# 2. Multivariate case — XTB currency portfolio

We build a portfolio from 3 exchange rates (EUR/PLN, USD/PLN, GBP/PLN) with weights matching the approximate revenue structure of XTB S.A.:
- **EUR/PLN: 50%** (main market — Europe)
- **USD/PLN: 30%** (US dollar markets)
- **GBP/PLN: 20%** (UK market)

Portfolio return: $R_p = w_1 R_{EUR} + w_2 R_{USD} + w_3 R_{GBP}$

In [ ]:
# ============================================================
# 2a-c. Currency portfolio — multivariate risk measures
# ============================================================

# Portfolio weights (3 currencies)
currency_cols = ['EUR/PLN', 'USD/PLN', 'GBP/PLN']
weights = np.array([0.50, 0.30, 0.20])

# Portfolio return
portfolio_returns = (log_returns[currency_cols] * weights).sum(axis=1)

# Covariance and correlation matrices
cov_matrix = log_returns[currency_cols].cov()
corr_matrix = log_returns[currency_cols].corr()

print("=" * 70)
print("2. MULTIVARIATE CASE — currency portfolio")
print("=" * 70)

print(f"\nPortfolio weights: EUR={weights[0]:.0%}, USD={weights[1]:.0%}, GBP={weights[2]:.0%}")

print("\n--- Correlation matrix ---")
display(corr_matrix.round(4))

print("\n--- Covariance matrix (×10⁶) ---")
display((cov_matrix * 1e6).round(4))

In [ ]:
# ============================================================
# 2a. Portfolio volatility measures
# ============================================================

# Portfolio volatility: σ_p = sqrt(w' Σ w)
sigma_p_analytical = np.sqrt(weights @ cov_matrix.values @ weights)
sigma_p_empirical = portfolio_returns.std()

print("--- 2a. Portfolio volatility measures ---\n")
print(f"σ portfolio (analytical, from covariance matrix): {sigma_p_analytical:.6f}")
print(f"σ portfolio (empirical, from portfolio series):      {sigma_p_empirical:.6f}")
print(f"σ annualizowane:                                  {sigma_p_empirical * np.sqrt(252):.4f}")
print(f"Portfolio variance:                               {portfolio_returns.var():.8f}")
print(f"Portfolio IQR:                                     {portfolio_returns.quantile(0.75) - portfolio_returns.quantile(0.25):.6f}")
print(f"Portfolio MAD:                                     {np.median(np.abs(portfolio_returns - portfolio_returns.median())):.6f}")
print(f"Portfolio skewness:                                {portfolio_returns.skew():.4f}")
print(f"Portfolio kurtosis (excess):                        {portfolio_returns.kurtosis():.4f}")

# Diversification effect
sum_weighted_sigmas = sum(w * log_returns[c].std() for w, c in zip(weights, currency_cols))
print(f"\nΣ(w_i × σ_i) (without diversification): {sum_weighted_sigmas:.6f}")
print(f"σ_portfolio (with diversification):      {sigma_p_empirical:.6f}")
print(f"Risk reduction through diversification: {(1 - sigma_p_empirical/sum_weighted_sigmas)*100:.1f}%")

In [ ]:
# ============================================================
# 2b. Portfolio quantiles / VaR
# ============================================================

print("--- 2b. Portfolio VaR and ES ---\n")

# Distribution fitting for the portfolio
pr = portfolio_returns.values
mu_p, sig_p = pr.mean(), pr.std()
df_tp, loc_tp, scale_tp = t_dist.fit(pr)

port_var_results = []
for alpha in alpha_levels:
    var_emp = np.percentile(pr, alpha * 100)
    var_norm = norm.ppf(alpha, loc=mu_p, scale=sig_p)
    var_t = t_dist.ppf(alpha, df_tp, loc=loc_tp, scale=scale_tp)
    es_emp = pr[pr <= var_emp].mean()
    
    port_var_results.append({
        'α': f'{alpha:.0%}',
        'VaR emp.': f'{var_emp:.6f}',
        'VaR Normal': f'{var_norm:.6f}',
        'VaR t-Student': f'{var_t:.6f}',
        'ES emp.': f'{es_emp:.6f}',
    })

port_var_df = pd.DataFrame(port_var_results)
port_var_df

In [ ]:
# ============================================================
# 2c. Portfolio density + bivariate distribution
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('XTB currency portfolio — density and bivariate distribution', fontsize=14, fontweight='bold')

# Portfolio histogram
ax = axes[0]
x_grid_p = np.linspace(pr.min(), pr.max(), 500)
ax.hist(pr, bins=60, density=True, alpha=0.5, color='steelblue', label='Empirical')
ax.plot(x_grid_p, norm.pdf(x_grid_p, mu_p, sig_p), 'r-', lw=2, label='Normal')
ax.plot(x_grid_p, t_dist.pdf(x_grid_p, df_tp, loc_tp, scale_tp), 'g--', lw=2, label='t-Student')
ax.set_title('Currency portfolio density')
ax.legend()
ax.grid(alpha=0.3)

# Bivariate scatter plot: EUR/PLN vs USD/PLN (two most important currencies)
ax = axes[1]
sc = ax.scatter(log_returns['EUR/PLN'], log_returns['USD/PLN'], 
                alpha=0.3, s=8, c=log_returns['GBP/PLN'], cmap='RdYlGn')
ax.set_xlabel('EUR/PLN log-return')
ax.set_ylabel('USD/PLN log-return')
ax.set_title('Bivariate distribution (color = GBP/PLN)')
ax.axhline(0, color='black', lw=0.5)
ax.axvline(0, color='black', lw=0.5)
plt.colorbar(sc, ax=ax, label='GBP/PLN')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Multivariate distribution function: P(R_EUR ≤ x, R_USD ≤ x, R_GBP ≤ x)
print("\nEmpirical multivariate distribution function F(x, x, x) = P(R_EUR ≤ x, R_USD ≤ x, R_GBP ≤ x):")
print("(all currencies simultaneously below threshold x)\n")
thresholds = [0, -0.005, -0.01, -0.02]
for x in thresholds:
    mask = (log_returns[currency_cols] <= x).all(axis=1)
    f_multi = mask.mean()
    # Comparison with product of marginals (independence test)
    f_product = np.prod([np.mean(log_returns[c] <= x) for c in currency_cols])
    print(f"  F({x:+.3f}, {x:+.3f}, {x:+.3f}) = {f_multi:.4f}  "
          f"| Product of marginals: {f_product:.4f}  "
          f"| Ratio: {f_multi/f_product:.2f}x" if f_product > 0 else f"  F({x}) = {f_multi:.4f}")


In [ ]:
# ============================================================
# 2c (cont.). Multivariate distribution function plots + portfolio density
# ============================================================
from scipy.stats import multivariate_normal, gaussian_kde

r_eur = log_returns['EUR/PLN'].values
r_usd = log_returns['USD/PLN'].values
r_gbp = log_returns['GBP/PLN'].values
n = len(r_eur)

# --- Grid for computing F(x, y) = P(R_EUR ≤ x, R_USD ≤ y) ---
grid_size = 80
x_vals = np.linspace(np.percentile(r_eur, 1), np.percentile(r_eur, 99), grid_size)
y_vals = np.linspace(np.percentile(r_usd, 1), np.percentile(r_usd, 99), grid_size)
X, Y = np.meshgrid(x_vals, y_vals)

# Empirical bivariate CDF
F_emp = np.zeros_like(X)
for i in range(grid_size):
    for j in range(grid_size):
        F_emp[i, j] = np.mean((r_eur <= X[i, j]) & (r_usd <= Y[i, j]))

# Theoretical CDF — bivariate normal distribution
mean_2d = np.array([r_eur.mean(), r_usd.mean()])
cov_2d = np.cov(r_eur, r_usd)
mv_norm = multivariate_normal(mean=mean_2d, cov=cov_2d)
F_norm = np.zeros_like(X)
for i in range(grid_size):
    for j in range(grid_size):
        F_norm[i, j] = mv_norm.cdf([X[i, j], Y[i, j]])

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('Multivariate distribution function and density — XTB currency portfolio', fontsize=14, fontweight='bold')

# === Plot 1: empirical CDF heatmap ===
ax = axes[0]
cf = ax.contourf(X * 100, Y * 100, F_emp, levels=20, cmap='viridis')
cs = ax.contour(X * 100, Y * 100, F_emp, levels=10, colors='white', linewidths=0.5, alpha=0.6)
ax.clabel(cs, fontsize=7, fmt='%.2f', colors='white')
plt.colorbar(cf, ax=ax, label='F(x, y)')
ax.set_xlabel('EUR/PLN (%)')
ax.set_ylabel('USD/PLN (%)')
ax.set_title('Empirical bivariate CDF')
ax.grid(alpha=0.2)

# === Plot 2: empirical vs normal contours ===
ax = axes[1]
levels = np.arange(0.05, 1.0, 0.1)
from matplotlib.lines import Line2D
cs1 = ax.contour(X * 100, Y * 100, F_emp, levels=levels, cmap='Blues', linewidths=1.5)
cs2 = ax.contour(X * 100, Y * 100, F_norm, levels=levels, cmap='Reds', linewidths=1.5, linestyles='--')
ax.clabel(cs1, fontsize=7, fmt='%.2f')
ax.set_xlabel('EUR/PLN (%)')
ax.set_ylabel('USD/PLN (%)')
ax.set_title('CDF contours: empirical (blue) vs normal (red)')
ax.legend([Line2D([0],[0], color='steelblue', lw=2),
           Line2D([0],[0], color='firebrick', lw=2, ls='--')],
          ['Empirical', 'Normal'], loc='upper left')
ax.grid(alpha=0.3)

# === Plot 3: portfolio density scatter (EUR vs USD, color = KDE) ===
ax = axes[2]
xy = np.vstack([r_eur, r_usd])
kde = gaussian_kde(xy)
density = kde(xy)
idx = density.argsort()  # sort so densest points are on top
sc = ax.scatter(r_eur[idx] * 100, r_usd[idx] * 100, c=density[idx], s=8, alpha=0.6, cmap='inferno')
plt.colorbar(sc, ax=ax, label='Density (KDE)')
ax.set_xlabel('EUR/PLN (%)')
ax.set_ylabel('USD/PLN (%)')
ax.set_title('Portfolio density — 2D scatter (EUR vs USD)')
ax.axhline(0, color='grey', lw=0.5)
ax.axvline(0, color='grey', lw=0.5)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Interpretation:")
print("• Left: heatmap of empirical joint CDF P(R_EUR ≤ x, R_USD ≤ y)")
print("• Center: empirical vs normal CDF contours — differences = deviations from Gaussian dependence")
print("• Right: scatter colored by KDE density — shows concentration and tails of the joint distribution")


In [ ]:
# ============================================================
# 2d. Comparison: portfolio vs individual variables
# ============================================================

print("=" * 70)
print("2d. COMPARISON — portfolio vs individual variables")
print("=" * 70)

comparison = []
# Individual currencies
for col in currency_cols:
    r = log_returns[col].values
    comparison.append({
        'Variable': col,
        'σ daily': f'{r.std():.6f}',
        'σ annual': f'{r.std()*np.sqrt(252):.4f}',
        'VaR 5% emp.': f'{np.percentile(r, 5):.6f}',
        'VaR 1% emp.': f'{np.percentile(r, 1):.6f}',
        'ES 5% emp.': f'{r[r <= np.percentile(r, 5)].mean():.6f}',
        'Skewness': f'{stats.skew(r):.4f}',
        'Kurtosis': f'{stats.kurtosis(r):.4f}',
    })

# Portfolio
comparison.append({
    'Variable': 'PORTFOLIO',
    'σ daily': f'{pr.std():.6f}',
    'σ annual': f'{pr.std()*np.sqrt(252):.4f}',
    'VaR 5% emp.': f'{np.percentile(pr, 5):.6f}',
    'VaR 1% emp.': f'{np.percentile(pr, 1):.6f}',
    'ES 5% emp.': f'{pr[pr <= np.percentile(pr, 5)].mean():.6f}',
    'Skewness': f'{stats.skew(pr):.4f}',
    'Kurtosis': f'{stats.kurtosis(pr):.4f}',
})

comp_df = pd.DataFrame(comparison)
print("\nRisk measures comparison — portfolio should have lower risk thanks to diversification:\n")
comp_df

---
# 2e. CFD portfolio — Gold, S&P 500, NASDAQ-100

As a CFD broker, XTB generates commission and spread revenue from client transaction volume. The most popular CFD instruments are:
- **Gold (XAUUSD)** — safe haven, high volatility during periods of uncertainty
- **S&P 500 (^GSPC)** — US market benchmark
- **NASDAQ-100 (^NDX)** — technology index, highest volatility

**Portfolio weights** reflect the approximate CFD volume structure on the XTB platform:
- Gold: 40% (most popular CFD instrument)
- S&P 500: 35%
- NASDAQ-100: 25%

**Key question:** How does the volatility of these instruments affect the VaR of XTB's exposure portfolio and the company's potential gains/losses? Higher volatility = higher spread + volume, but also higher risk of negative client balances.

In [ ]:
# ============================================================
# 2e. CFD portfolio — data download and portfolio construction
# ============================================================

cfd_tickers = {
    'GC=F':  'Gold (XAU)',
    '^GSPC': 'S&P 500',
    '^NDX':  'NASDAQ-100'
}

cfd_data = yf.download(list(cfd_tickers.keys()), start='2022-01-01', end='2025-12-31')['Close']
cfd_data.columns = [cfd_tickers.get(c, c) for c in cfd_data.columns]
cfd_data = cfd_data.dropna()

cfd_returns = np.log(cfd_data / cfd_data.shift(1)).dropna()

# CFD portfolio weights
cfd_cols = ['Gold (XAU)', 'S&P 500', 'NASDAQ-100']
cfd_weights = np.array([0.40, 0.35, 0.25])

cfd_portfolio_returns = (cfd_returns[cfd_cols] * cfd_weights).sum(axis=1)

# Common period with currency portfolio
common_idx = cfd_portfolio_returns.index.intersection(portfolio_returns.index)
cfd_pr = cfd_portfolio_returns.loc[common_idx].values
fx_pr = portfolio_returns.loc[common_idx].values

print(f"CFD data period: {cfd_data.index[0].date()} — {cfd_data.index[-1].date()}")
print(f"Observations (log-returns): {len(cfd_returns)}")
print(f"Common period with currency portfolio: {len(common_idx)} days")

print(f"\nDescriptive statistics of CFD log-returns (daily):")
cfd_returns.describe().T

In [ ]:
# ============================================================
# 2e (cont.). CFD portfolio risk analysis + comparison with currency portfolio
# ============================================================
from scipy.stats import gaussian_kde

# --- CFD correlation and covariance matrices ---
cfd_cov = cfd_returns[cfd_cols].cov()
cfd_corr = cfd_returns[cfd_cols].corr()

print("=" * 70)
print("2e. CFD PORTFOLIO — RISK MEASURES")
print("=" * 70)

print("\n--- CFD correlation matrix ---")
display(cfd_corr.round(4))

# --- Volatility ---
cfd_pr_all = cfd_portfolio_returns.values
sigma_cfd = cfd_pr_all.std()
sigma_cfd_ann = sigma_cfd * np.sqrt(252)

sum_weighted_sigmas_cfd = sum(w * cfd_returns[c].std() for w, c in zip(cfd_weights, cfd_cols))
diversification_cfd = (1 - sigma_cfd / sum_weighted_sigmas_cfd) * 100

print(f"\n--- CFD portfolio volatility ---")
print(f"σ daily:       {sigma_cfd:.6f}")
print(f"σ annual:        {sigma_cfd_ann:.4f}")
print(f"Risk reduction through diversification: {diversification_cfd:.1f}%")

# --- VaR and ES ---
mu_cfd, sig_cfd = cfd_pr_all.mean(), cfd_pr_all.std()
df_cfd, loc_cfd, scale_cfd = t_dist.fit(cfd_pr_all)

print(f"\n--- CFD portfolio VaR and ES ---")
cfd_var_rows = []
for alpha in [0.01, 0.05, 0.10]:
    var_emp = np.percentile(cfd_pr_all, alpha * 100)
    var_norm = norm.ppf(alpha, loc=mu_cfd, scale=sig_cfd)
    var_t = t_dist.ppf(alpha, df_cfd, loc=loc_cfd, scale=scale_cfd)
    es_emp = cfd_pr_all[cfd_pr_all <= var_emp].mean()
    cfd_var_rows.append({
        'α': f'{alpha:.0%}',
        'VaR emp.': f'{var_emp:.6f}',
        'VaR Normal': f'{var_norm:.6f}',
        'VaR t-Student': f'{var_t:.6f}',
        'ES emp.': f'{es_emp:.6f}',
    })
display(pd.DataFrame(cfd_var_rows))
rho = np.corrcoef(fx_pr, cfd_pr)[0, 1]
# ============================================================
# COMPARISON: CFD portfolio vs currency portfolio
# ============================================================
print("\n" + "=" * 70)
print("COMPARISON: CFD portfolio vs currency portfolio")
print("=" * 70)

comp_portfolios = []
for name, ret, w, cols, ret_df in [
    ('Currency (FX)', fx_pr, weights, currency_cols, log_returns),
    ('CFD', cfd_pr, cfd_weights, cfd_cols, cfd_returns)
]:
    comp_portfolios.append({
        'Portfolio': name,
        'σ daily': f'{ret.std():.6f}',
        'σ annual': f'{ret.std()*np.sqrt(252):.4f}',
        'Daily mean': f'{ret.mean():.6f}',
        'VaR 1%': f'{np.percentile(ret, 1):.6f}',
        'VaR 5%': f'{np.percentile(ret, 5):.6f}',
        'ES 1%': f'{ret[ret <= np.percentile(ret, 1)].mean():.6f}',
        'Skewness': f'{stats.skew(ret):.4f}',
        'Kurtosis': f'{stats.kurtosis(ret):.4f}',
    })
display(pd.DataFrame(comp_portfolios).set_index('Portfolio'))

# --- Volatility of individual CFD instruments vs currencies ---
print("\n--- Volatility of individual instruments (σ annual) ---")
vol_rows = []
for col in currency_cols:
    r = log_returns[col].dropna().values
    vol_rows.append({'Instrument': col, 'Typ': 'FX', 'σ annual': f'{r.std()*np.sqrt(252):.4f}',
                     'Kurtosis': f'{stats.kurtosis(r):.2f}'})
for col in cfd_cols:
    r = cfd_returns[col].dropna().values
    vol_rows.append({'Instrument': col, 'Typ': 'CFD', 'σ annual': f'{r.std()*np.sqrt(252):.4f}',
                     'Kurtosis': f'{stats.kurtosis(r):.2f}'})
display(pd.DataFrame(vol_rows).set_index('Instrument'))



# --- Interpretation ---
print("\n--- Interpretation of impact on XTB ---")
vol_ratio = cfd_pr_all.std() / portfolio_returns.std()
print(f"• CFD portfolio volatility is {vol_ratio:.1f}x higher than the currency portfolio")
print(f"• Correlation between portfolios: ρ = {rho:.3f}")
print(f"  → {'Low correlation — diversification of XTB risk sources' if abs(rho) < 0.3 else 'Moderate/high correlation — risk accumulates'}")
print(f"• CFD portfolio VaR 1% ({np.percentile(cfd_pr, 1)*100:.2f}%) vs currency ({np.percentile(fx_pr, 1)*100:.2f}%)")
print(f"  → CFD generuje {'greater' if abs(np.percentile(cfd_pr, 1)) > abs(np.percentile(fx_pr, 1)) else 'smaller'} extreme loss risk")
print(f"• For XTB: high CFD volatility is a double-edged sword:")
print(f"  - GAIN: higher vol → higher spreads, more client transactions, higher commissions")
print(f"  - RISK: higher vol → more negative client balances, higher capital requirements (KNF)")
print(f"  - Optimal volatility range: moderate vol maximizes net profit")

In [ ]:
# ============================================================
# 2f. Comparison: FX spot portfolio vs leveraged 5:1 currency CFD
# ============================================================
leverage = 5

leveraged_returns = portfolio_returns * leverage
lev_pr = leveraged_returns.values

spot_series = pd.Series(portfolio_returns.values, index=portfolio_returns.index)
lev_series = pd.Series(lev_pr, index=portfolio_returns.index)

# Rolling annualized sigma
fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle(f'Currency portfolio: FX spot vs CFD with leverage {leverage}:1', fontsize=14, fontweight='bold')

window = 21
(spot_series.rolling(window).std() * np.sqrt(252) * 100).plot(ax=ax, color='steelblue', lw=1, label='σ spot')
(lev_series.rolling(window).std() * np.sqrt(252) * 100).plot(ax=ax, color='indianred', lw=1, label=f'σ CFD {leverage}:1')
ax.set_title(f'Rolling annualized σ ({window}d)')
ax.set_ylabel('Annual volatility (%)')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


---
# 3. Extreme risk (EVT — Extreme Value Theory)

For the **XTB.WA** variable (share price) we apply two approaches:
- **Block Maxima (GEV):** we divide data into monthly blocks, take loss maxima → fit the GEV distribution
- **Peaks Over Threshold (GPD):** we take losses exceeding a high threshold (95% quantile) → fit the Generalized Pareto Distribution (GPD)

Both methods allow modeling the loss distribution tail and estimating VaR/ES for extremely low probabilities.

In [ ]:
# ============================================================
# 3a. EVT — Block Maxima (GEV) for XTB.WA
# ============================================================

losses_xtb = -log_returns['XTB.WA']  # losses = negative returns

# Block Maxima — maximum daily losses in each month
losses_monthly_max = losses_xtb.resample('ME').max().dropna()

# GEV fitting (Generalized Extreme Value)
gev_params = genextreme.fit(losses_monthly_max)
c_gev, loc_gev, scale_gev = gev_params

print("=" * 70)
print("3. EXTREME RISK — XTB.WA")
print("=" * 70)

print(f"\n--- 3a. Block Maxima (GEV) ---")
print(f"Number of blocks (months): {len(losses_monthly_max)}")
print(f"GEV parameters: ξ (shape)={c_gev:.4f}, μ (loc)={loc_gev:.6f}, σ (scale)={scale_gev:.6f}")
print(f"ξ < 0 → Weibull distribution (bounded tail)")
print(f"ξ = 0 → Gumbel distribution")
print(f"ξ > 0 → Fréchet distribution (heavy tail)")

# VaR from GEV
for p in [0.95, 0.99, 0.999]:
    var_gev = genextreme.ppf(p, c_gev, loc=loc_gev, scale=scale_gev)
    print(f"  VaR (loss) {p:.1%} from GEV: {var_gev:.5f} ({var_gev*100:.2f}%)")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
fig.suptitle('EVT — XTB.WA: Block Maxima (GEV) and Peaks Over Threshold (GPD)', fontsize=14, fontweight='bold')

ax = axes[0]
ax.hist(losses_monthly_max, bins=30, density=True, alpha=0.5, color='indianred', label='Monthly maxima')
x_gev = np.linspace(losses_monthly_max.min(), losses_monthly_max.max()*1.3, 200)
ax.plot(x_gev, genextreme.pdf(x_gev, *gev_params), 'b-', lw=2, label='GEV')
ax.set_title('Block Maxima — GEV')
ax.set_xlabel('Maximum daily loss')
ax.legend()
ax.grid(alpha=0.3)

# QQ-plot GEV
ax = axes[1]
theoretical_q = genextreme.ppf(np.linspace(0.01, 0.99, len(losses_monthly_max)), *gev_params)
empirical_q = np.sort(losses_monthly_max.values)
ax.scatter(theoretical_q, empirical_q, alpha=0.6, s=20, color='indianred')
lims = [min(theoretical_q.min(), empirical_q.min()), max(theoretical_q.max(), empirical_q.max())]
ax.plot(lims, lims, 'k--', lw=1)
ax.set_xlabel('Theoretical quantiles (GEV)')
ax.set_ylabel('Empirical quantiles')
ax.set_title('QQ-plot — GEV')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 3a (cd). EVT — Peaks Over Threshold (GPD) for XTB.WA
# ============================================================
from scipy.optimize import minimize

# Threshold: 92.5% loss quantile
threshold = np.percentile(losses_xtb, 92.5)
exceedances = losses_xtb[losses_xtb > threshold] - threshold  # excesses above threshold

# --- GPD fitting via MLE (scipy) ---
gpd_params = genpareto.fit(exceedances, floc=0)
c_gpd, _, scale_gpd = gpd_params

n_total = len(losses_xtb)
n_exceed = len(exceedances)

print(f"\n--- 3a (cd). Peaks Over Threshold (GPD) ---")
print(f"Threshold u (92.5% loss quantile): {threshold:.5f} ({threshold*100:.2f}%)")
print(f"Number of exceedances: {n_exceed} / {n_total} ({n_exceed/n_total*100:.1f}%)")
print(f"GPD parameters (MLE): ξ (shape)={c_gpd:.4f}, σ (scale)={scale_gpd:.6f}")

# --- Estymacja QML (Quasi-Maximum Likelihood) ---
# QML: maximize GPD log-likelihood, but use sandwich (robust)
# standard errors robust to possible model misspecification.

def gpd_negloglik(params, data):
    """Negative GPD log-likelihood with numerical safeguards."""
    xi, sigma = params
    if sigma <= 0:
        return 1e10
    n = len(data)
    if abs(xi) < 1e-8:  # case xi -> 0 (exponential distribution)
        return n * np.log(sigma) + np.sum(data) / sigma
    else:
        z = 1 + xi * data / sigma
        if np.any(z <= 0):
            return 1e10
        return n * np.log(sigma) + (1 + 1/xi) * np.sum(np.log(z))

# QML estimation — start from MLE as initial point
exc_arr = exceedances.values
res_qml = minimize(gpd_negloglik, x0=[c_gpd, scale_gpd], args=(exc_arr,),
                   method='Nelder-Mead', options={'maxiter': 10000, 'xatol': 1e-10, 'fatol': 1e-10})
xi_qml, sigma_qml = res_qml.x

# Robust (sandwich) standard errors for QML
from scipy.optimize import approx_fprime

def gpd_score_i(params, x_i):
    """Log-likelihood gradient for a single observation."""
    xi, sigma = params
    if sigma <= 0:
        return np.array([0.0, 0.0])
    if abs(xi) < 1e-8:
        dl_dsigma = -1/sigma + x_i / sigma**2
        dl_dxi = 0.0
    else:
        z = 1 + xi * x_i / sigma
        if z <= 0:
            return np.array([0.0, 0.0])
        dl_dxi = -(1 + 1/xi) * x_i / (sigma * z) + np.log(z) / xi**2
        dl_dsigma = -1/sigma + (1 + 1/xi) * xi * x_i / (sigma**2 * z)
    return np.array([dl_dxi, dl_dsigma])

# Matrix B (meat) — variance of gradients
scores = np.array([gpd_score_i([xi_qml, sigma_qml], x) for x in exc_arr])
B = scores.T @ scores / n_exceed

# Matrix A (bread) — numerical Hessian
eps_h = 1e-5
H = np.zeros((2, 2))
for i in range(2):
    def f_i(params, idx=i):
        return np.mean([gpd_score_i(params, x)[idx] for x in exc_arr])
    H[i, :] = approx_fprime([xi_qml, sigma_qml], f_i, eps_h)
A = -H

# Sandwich: V = A^{-1} B A^{-1} / n
try:
    A_inv = np.linalg.inv(A)
    V_sandwich = A_inv @ B @ A_inv / n_exceed
    se_qml = np.sqrt(np.diag(V_sandwich))
except np.linalg.LinAlgError:
    se_qml = np.array([np.nan, np.nan])

print(f"\n--- Estymacja QML (Quasi-Maximum Likelihood) ---")
print(f"GPD parameters (QML): ξ={xi_qml:.4f} (SE={se_qml[0]:.4f}), σ={sigma_qml:.6f} (SE={se_qml[1]:.6f})")
print(f"Robust sandwich standard errors (robust to misspecification)")
print(f"MLE vs QML comparison:  Δξ={abs(c_gpd - xi_qml):.6f},  Δσ={abs(scale_gpd - sigma_qml):.6f}")

# VaR and ES from GPD (MLE and QML)
print(f"\nVaR / ES — MLE vs QML comparison:")
for p in [0.95, 0.99, 0.999]:
    for label, xi, sig in [('MLE', c_gpd, scale_gpd), ('QML', xi_qml, sigma_qml)]:
        if xi != 0:
            var_g = threshold + (sig / xi) * ((n_total / n_exceed * (1 - p))**(-xi) - 1)
            es_g = var_g / (1 - xi) + (sig - xi * threshold) / (1 - xi)
        else:
            var_g = threshold + sig * np.log(n_total / n_exceed * (1 - p))
            es_g = var_g + sig
        print(f"  [{label}] VaR {p:.1%}: {var_g:.5f} ({var_g*100:.2f}%)  |  ES: {es_g:.5f} ({es_g*100:.2f}%)")

# ============================================================
# Ploty GPD — 3 panele
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('EVT — Peaks Over Threshold (GPD) for XTB.WA', fontsize=14, fontweight='bold')

# --- Plot 1: Histogram + PDF ---
ax = axes[0]
ax.hist(exceedances, bins=40, density=True, alpha=0.5, color='indianred', label='Exceedances')
x_gpd = np.linspace(0, exceedances.max()*1.2, 200)
ax.plot(x_gpd, genpareto.pdf(x_gpd, c_gpd, loc=0, scale=scale_gpd), 'b-', lw=2, label='GPD (MLE)')
ax.plot(x_gpd, genpareto.pdf(x_gpd, xi_qml, loc=0, scale=sigma_qml), 'g--', lw=2, label='GPD (QML)')
ax.set_title(f'Peaks Over Threshold (threshold={threshold:.4f})')
ax.set_xlabel('Excess above threshold')
ax.legend()
ax.grid(alpha=0.3)

# --- Plot 2: Mean Excess Plot (scatter + theoretical GPD line) ---
# For GPD: e(u) = E[X-u | X>u] = (σ + ξ·u) / (1-ξ)  — linear function in u
ax = axes[1]
thresholds_range = np.linspace(np.percentile(losses_xtb, 80), np.percentile(losses_xtb, 99), 80)
mean_excess = []
thresh_valid = []
for u in thresholds_range:
    exc = losses_xtb[losses_xtb > u] - u
    if len(exc) > 5:
        mean_excess.append(exc.mean())
        thresh_valid.append(u)

mean_excess = np.array(mean_excess)
thresh_valid = np.array(thresh_valid)

# Empirical scatter
ax.scatter(thresh_valid, mean_excess, s=20, color='indianred', alpha=0.7, zorder=3, label='Empirical')

# Theoretical GPD line (MLE): e(u) = (σ_GPD + ξ·(u - threshold_base)) / (1-ξ)
# Where σ_GPD and ξ are GPD parameters fitted at threshold
# For threshold u > threshold: σ(u) = σ + ξ·(u - threshold), so e(u) = (σ + ξ·(u-threshold)) / (1-ξ)
if c_gpd != 1:
    me_theoretical = (scale_gpd + c_gpd * (thresh_valid - threshold)) / (1 - c_gpd)
else:
    me_theoretical = np.full_like(thresh_valid, np.nan)

ax.plot(thresh_valid, me_theoretical, 'b-', lw=2, label=f'Theoretical GPD (MLE)')

# Correlation coefficient: empirical mean excess vs theoretical line
mask = np.isfinite(me_theoretical) & np.isfinite(mean_excess)
corr = np.corrcoef(mean_excess[mask], me_theoretical[mask])[0, 1]
print(f"\n--- Mean Excess Plot ---")
print(f"Correlation coefficient (empirical ME vs theoretical GPD): r = {corr:.4f}")

# Empirical linear regression (OLS) for comparison
from numpy.polynomial.polynomial import polyfit
b0, b1 = polyfit(thresh_valid, mean_excess, 1)
me_ols = b0 + b1 * thresh_valid
corr_ols = np.corrcoef(mean_excess, me_ols)[0, 1]
ax.plot(thresh_valid, me_ols, 'g--', lw=1.5, label=f'OLS regression (r={corr_ols:.4f})')

ax.axvline(threshold, color='blue', linestyle=':', alpha=0.5, label=f'Selected threshold = {threshold:.4f}')
ax.set_title(f'Mean Excess Plot (r={corr:.4f})')
ax.set_xlabel('Threshold u')
ax.set_ylabel('Mean excess E[X-u | X>u]')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

print(f"OLS regression: intercept={b0:.6f}, slope={b1:.4f}, r={corr_ols:.4f}")

# --- Plot 3: Empirical CDF vs GPD (MLE and QML) ---
ax = axes[2]
sorted_exc = np.sort(exceedances.values)
ecdf_exc = np.arange(1, len(sorted_exc) + 1) / len(sorted_exc)
x_cdf = np.linspace(0, sorted_exc.max() * 1.1, 300)

ax.step(sorted_exc, ecdf_exc, color='steelblue', lw=2.5, label='ECDF (empirical)')
ax.plot(x_cdf, genpareto.cdf(x_cdf, c_gpd, loc=0, scale=scale_gpd),
        'r-', lw=2, label=f'GPD MLE (ξ={c_gpd:.3f}, σ={scale_gpd:.4f})')
ax.plot(x_cdf, genpareto.cdf(x_cdf, xi_qml, loc=0, scale=sigma_qml),
        'g--', lw=2, label=f'GPD QML (ξ={xi_qml:.3f}, σ={sigma_qml:.4f})')

# Mark max KS deviation for MLE
idx_max = np.argmax(np.abs(ecdf_exc - genpareto.cdf(sorted_exc, c_gpd, loc=0, scale=scale_gpd)))
ks_stat_val = abs(ecdf_exc[idx_max] - genpareto.cdf(sorted_exc[idx_max], c_gpd, loc=0, scale=scale_gpd))
ax.vlines(sorted_exc[idx_max],
          genpareto.cdf(sorted_exc[idx_max], c_gpd, loc=0, scale=scale_gpd), ecdf_exc[idx_max],
          color='orange', lw=2, ls='--', label=f'max |D| KS = {ks_stat_val:.4f}')

ax.set_title('Empirical CDF vs GPD')
ax.set_xlabel('Excess above threshold')
ax.set_ylabel('CDF')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 3b. K-S test — comparison of extreme distributions
#     Weibull, Gumbel, Fréchet and GEV for Block Maxima
#     + GPD for Peaks Over Threshold
# ============================================================
from scipy.stats import weibull_min, gumbel_r, invweibull  # invweibull = Fréchet

print("=" * 70)
print("3b. KOLMOGOROV-SMIRNOV TEST — extreme distributions")
print("=" * 70)

bm = losses_monthly_max.values

# --- Fitting 4 distributions to Block Maxima ---
fits_bm = {}

# 1) GEV (general — already fitted earlier)
fits_bm['GEV'] = {'params': gev_params, 'dist': genextreme}

# 2) Gumbel (GEV z ξ=0)
gumbel_params = gumbel_r.fit(bm)
fits_bm['Gumbel'] = {'params': gumbel_params, 'dist': gumbel_r}

# 3) Weibull (GEV with ξ<0 — bounded tail)
weibull_params = weibull_min.fit(bm)
fits_bm['Weibull'] = {'params': weibull_params, 'dist': weibull_min}

# 4) Fréchet (GEV with ξ>0 — heavy tail)
frechet_params = invweibull.fit(bm)
fits_bm['Fréchet'] = {'params': frechet_params, 'dist': invweibull}

# --- KS tests ---
print(f"\n{'─'*70}")
print(f"  Block Maxima (n={len(bm)})")
print(f"{'─'*70}")

ks_rows = []
for name, info in fits_bm.items():
    ks_stat, ks_p = kstest(bm, info['dist'].cdf, args=info['params'])
    ks_rows.append({
        'Distribution': name,
        'KS stat': round(ks_stat, 4),
        'p-value': round(ks_p, 4),
        'Conclusion (α=0.05)': 'OK — no grounds to reject' if ks_p > 0.05 else 'REJECTED'
    })
    info['ks_stat'] = ks_stat
    info['ks_p'] = ks_p

ks_df = pd.DataFrame(ks_rows).set_index('Distribution')
print(ks_df.to_string())

best_bm = min(fits_bm.items(), key=lambda x: x[1]['ks_stat'])
print(f"\n→ Best fit: {best_bm[0]} (KS={best_bm[1]['ks_stat']:.4f}, p={best_bm[1]['ks_p']:.4f})")

# --- KS for GPD (Peaks Over Threshold) ---
print(f"\n{'─'*70}")
print(f"  Peaks Over Threshold (threshold={threshold:.4f}, n={len(exceedances)})")
print(f"{'─'*70}")

ks_gpd_stat, ks_gpd_p = kstest(exceedances, 'genpareto', args=(c_gpd, 0, scale_gpd))
print(f"  GPD:  KS stat = {ks_gpd_stat:.4f},  p-value = {ks_gpd_p:.4f}  →  "
      f"{'OK' if ks_gpd_p > 0.05 else 'REJECTED'}")

# === Plots: empirical CDF vs 4 distributions ===
colors = {'GEV': 'red', 'Gumbel': 'darkorange', 'Weibull': 'green', 'Fréchet': 'purple'}

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('K-S test — comparison of extreme distributions', fontsize=14, fontweight='bold')

# Plot 1: CDF Block Maxima — all 4 distributions
ax = axes[0]
sorted_bm = np.sort(bm)
ecdf_bm = np.arange(1, len(sorted_bm) + 1) / len(sorted_bm)
x_grid = np.linspace(sorted_bm.min() * 0.9, sorted_bm.max() * 1.1, 300)

ax.step(sorted_bm, ecdf_bm, color='steelblue', lw=2.5, label='Empirical')
for name, info in fits_bm.items():
    ax.plot(x_grid, info['dist'].cdf(x_grid, *info['params']),
            color=colors[name], lw=1.8, ls='--' if name != best_bm[0] else '-',
            label=f"{name} (p={info['ks_p']:.3f})")
ax.set_title('CDF — Block Maxima')
ax.set_xlabel('Maximum monthly loss')
ax.set_ylabel('CDF')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Plot 2: PDF Block Maxima — all 4 distributions
ax = axes[1]
ax.hist(bm, bins=20, density=True, alpha=0.4, color='steelblue', label='Empirical')
for name, info in fits_bm.items():
    ax.plot(x_grid, info['dist'].pdf(x_grid, *info['params']),
            color=colors[name], lw=1.8, ls='--' if name != best_bm[0] else '-',
            label=f"{name}")
ax.set_title('Density — Block Maxima')
ax.set_xlabel('Maximum monthly loss')
ax.set_ylabel('Density')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Plot 3: CDF GPD
ax = axes[2]
sorted_exc = np.sort(exceedances.values)
ecdf_exc = np.arange(1, len(sorted_exc) + 1) / len(sorted_exc)
x_gpd = np.linspace(0, sorted_exc.max() * 1.1, 300)
ax.step(sorted_exc, ecdf_exc, color='steelblue', lw=2.5, label='Empirical')
ax.plot(x_gpd, genpareto.cdf(x_gpd, c_gpd, loc=0, scale=scale_gpd),
        'r-', lw=2, label=f'GPD (p={ks_gpd_p:.3f})')
idx_max = np.argmax(np.abs(ecdf_exc - genpareto.cdf(sorted_exc, c_gpd, loc=0, scale=scale_gpd)))
ax.vlines(sorted_exc[idx_max],
          genpareto.cdf(sorted_exc[idx_max], c_gpd, loc=0, scale=scale_gpd), ecdf_exc[idx_max],
          color='green', lw=2, ls='--', label=f'max |D| = {ks_gpd_stat:.4f}')
ax.set_title('CDF — GPD (Peaks Over Threshold)')
ax.set_xlabel('Excess above threshold')
ax.set_ylabel('CDF')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Parameters of fitted distributions
print("\nParameters of fitted distributions (Block Maxima):")
print(f"  GEV:     ξ={gev_params[0]:.4f}, μ={gev_params[1]:.6f}, σ={gev_params[2]:.6f}")
print(f"  Gumbel:  μ={gumbel_params[0]:.6f}, σ={gumbel_params[1]:.6f}")
print(f"  Weibull: c={weibull_params[0]:.4f}, loc={weibull_params[1]:.6f}, scale={weibull_params[2]:.6f}")
print(f"  Fréchet: α={frechet_params[0]:.4f}, loc={frechet_params[1]:.6f}, scale={frechet_params[2]:.6f}")
print(f"\nNote: GEV is a generalization — Gumbel (ξ=0), Weibull (ξ<0), Fréchet (ξ>0)")

---
# 4. Interpretation of results

### Univariate case (section 1)
- **Volatility:** XTB.WA shares have the highest volatility (standard deviation, IQR) among the analyzed variables, which is typical — stocks are riskier than exchange rates. Among currencies, USD/PLN usually shows the highest volatility.
- **Quantiles / VaR:** The t-Student distribution better models distribution tails (fat tails), giving higher VaR estimates than the Normal distribution. This is consistent with observed excess kurtosis (kurtosis > 0 for all variables).
- **Distribution functions:** F(0) ≈ 50% for currency rates (symmetry of log-returns around zero), meaning the probability of loss on a given day is close to 50%. The t-Student distribution fits empirical data better, especially in the tails.

### Multivariate case (section 2)
- **Diversification:** The currency portfolio has lower volatility than the weighted average volatility of individual currencies — the diversification effect arises from imperfect correlation between rates.
- **Correlations:** Exchange rates against PLN are strongly correlated (common denominator — PLN), which limits the diversification effect.
- **Multivariate distribution function:** The ratio of the joint distribution function to the product of marginals > 1 indicates tail dependence — extremely negative currency moves tend to occur simultaneously.

### Extreme risk (section 3)
- **GEV:** The shape parameter ξ indicates tail heaviness: ξ > 0 means a heavy tail (Fréchet distribution), suggesting that extremely large losses on XTB shares are more likely than a Normal distribution would imply.
- **GPD:** The Mean Excess Plot helps verify threshold selection — a linear trend confirms the validity of the GPD model. VaR estimates from EVT are typically higher than from the Normal distribution, which is crucial for extreme risk management.
- **Conclusion for XTB:** As an FX/CFD broker, XTB is exposed to extreme market movements that can cause client losses exceeding deposits (negative balance risk). EVT models provide better estimates of capital required to cover such scenarios.

---
# 5. Backtesting

We verify VaR model quality using two tests:

**Case 1:** VaR 1% for EUR/PLN (univariate, Normal vs t-Student distribution)  
**Case 2:** VaR 1% for the currency portfolio (multivariate)

Method: rolling window (250 days) — estimate parameters from the window, forecast VaR for the next day, count violations.

**Kupiec test (POF):** H0: violation frequency = α (expected). LR test.  
**Christoffersen test:** H0: violations are independent and have the correct frequency.

In [ ]:
# ============================================================
# 5. Backtesting VaR — rolling window
# ============================================================

def kupiec_pof_test(violations, n_obs, alpha):
    """Test Kupca (Proportion of Failures) — LR test."""
    x = violations  # number of violations
    p_hat = x / n_obs if n_obs > 0 else 0
    if x == 0 or x == n_obs:
        return np.nan, np.nan
    lr = -2 * (np.log((1-alpha)**(n_obs-x) * alpha**x) - 
               np.log((1-p_hat)**(n_obs-x) * p_hat**x))
    p_value = 1 - stats.chi2.cdf(lr, df=1)
    return lr, p_value

def christoffersen_test(hit_series, alpha):
    """Christoffersen test (independence + coverage)."""
    hits = hit_series.astype(int).values
    n = len(hits)
    
    # Transition matrix
    n00 = n01 = n10 = n11 = 0
    for i in range(1, n):
        if hits[i-1] == 0 and hits[i] == 0: n00 += 1
        elif hits[i-1] == 0 and hits[i] == 1: n01 += 1
        elif hits[i-1] == 1 and hits[i] == 0: n10 += 1
        else: n11 += 1
    
    # LR independence
    if n01+n00 == 0 or n10+n11 == 0 or n01 == 0 or n10 == 0:
        return np.nan, np.nan
    
    p01 = n01 / (n00 + n01)
    p11 = n11 / (n10 + n11)
    p_hat = (n01 + n11) / n
    
    if p_hat == 0 or p_hat == 1 or p01 == 0 or p01 == 1:
        return np.nan, np.nan
    
    lr_ind = -2 * (np.log((1-p_hat)**(n00+n10) * p_hat**(n01+n11)) -
                   np.log((1-p01)**n00 * p01**n01 * (1-p11)**n10 * p11**n11))
    
    # LR coverage (Kupiec)
    x = n01 + n11
    lr_pof = -2 * (np.log((1-alpha)**(n-x) * alpha**x) -
                   np.log((1-p_hat)**(n-x) * p_hat**x))
    
    lr_cc = lr_ind + lr_pof
    p_value = 1 - stats.chi2.cdf(lr_cc, df=2)
    return lr_cc, p_value

# ---------- Backtesting ----------
window = 250
alpha_bt = 0.01

# Case 1: EUR/PLN (univariate)
# Case 2: Currency portfolio
series_dict = {
    'EUR/PLN (Normal)': ('EUR/PLN', 'norm'),
    'EUR/PLN (t-Student)': ('EUR/PLN', 't'),
    'Portfolio (Normal)': ('portfolio', 'norm'),
    'Portfolio (t-Student)': ('portfolio', 't'),
}

bt_results_all = {}

for label, (var_name, dist_type) in series_dict.items():
    if var_name == 'portfolio':
        returns_series = portfolio_returns
    else:
        returns_series = log_returns[var_name]
    
    var_forecasts = []
    actual_returns = []
    
    for i in range(window, len(returns_series)):
        window_data = returns_series.iloc[i-window:i].values
        
        if dist_type == 'norm':
            mu_w, sig_w = window_data.mean(), window_data.std()
            var_forecast = norm.ppf(alpha_bt, loc=mu_w, scale=sig_w)
        else:  # t-Student
            df_w, loc_w, scale_w = t_dist.fit(window_data)
            var_forecast = t_dist.ppf(alpha_bt, df_w, loc=loc_w, scale=scale_w)
        
        var_forecasts.append(var_forecast)
        actual_returns.append(returns_series.iloc[i])
    
    var_forecasts = np.array(var_forecasts)
    actual_returns = np.array(actual_returns)
    violations = actual_returns < var_forecasts
    
    bt_results_all[label] = {
        'var_forecasts': var_forecasts,
        'actual_returns': actual_returns,
        'violations': violations,
        'dates': returns_series.index[window:],
    }

print("=" * 70)
print("5. BACKTESTING — VaR 1% (rolling window 250 days)")
print("=" * 70)

bt_summary = []
for label, res in bt_results_all.items():
    n_obs = len(res['actual_returns'])
    n_viol = res['violations'].sum()
    rate = n_viol / n_obs
    
    lr_kupiec, p_kupiec = kupiec_pof_test(n_viol, n_obs, alpha_bt)
    lr_chris, p_chris = christoffersen_test(pd.Series(res['violations']), alpha_bt)
    
    bt_summary.append({
        'Model': label,
        'N obs.': n_obs,
        'Violations': n_viol,
        'Rate': f'{rate:.3%}',
        'Expected': f'{alpha_bt:.1%}',
        'Kupiec LR': f'{lr_kupiec:.3f}' if not np.isnan(lr_kupiec) else 'N/A',
        'Kupiec p-val': f'{p_kupiec:.4f}' if not np.isnan(p_kupiec) else 'N/A',
        'Christ. LR': f'{lr_chris:.3f}' if not np.isnan(lr_chris) else 'N/A',
        'Christ. p-val': f'{p_chris:.4f}' if not np.isnan(p_chris) else 'N/A',
    })

bt_df = pd.DataFrame(bt_summary)
print("\np-value > 0.05 → VaR model is acceptable (no grounds to reject)\n")
bt_df

In [ ]:
# Ploty backtestingu
fig, axes = plt.subplots(2, 1, figsize=(18, 10))
fig.suptitle('Backtesting VaR 1% — rolling window 250 days', fontsize=14, fontweight='bold')

# Case 1: EUR/PLN
ax = axes[0]
res_n = bt_results_all['EUR/PLN (Normal)']
res_t = bt_results_all['EUR/PLN (t-Student)']
dates = res_n['dates']

ax.plot(dates, res_n['actual_returns'], color='steelblue', lw=0.5, alpha=0.7, label='Log-return EUR/PLN')
ax.plot(dates, res_n['var_forecasts'], color='red', lw=1, label='VaR 1% (Normal)')
ax.plot(dates, res_t['var_forecasts'], color='green', lw=1, linestyle='--', label='VaR 1% (t-Student)')

# Mark violations
viol_mask_n = res_n['violations']
ax.scatter(dates[viol_mask_n], res_n['actual_returns'][viol_mask_n], 
           color='red', s=20, zorder=5, label=f'Normal violations ({viol_mask_n.sum()})')
ax.set_title('Case 1: EUR/PLN — VaR 1%')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Case 2: Portfolio
ax = axes[1]
res_pn = bt_results_all['Portfolio (Normal)']
res_pt = bt_results_all['Portfolio (t-Student)']
dates_p = res_pn['dates']

ax.plot(dates_p, res_pn['actual_returns'], color='steelblue', lw=0.5, alpha=0.7, label='Portfolio log-return')
ax.plot(dates_p, res_pn['var_forecasts'], color='red', lw=1, label='VaR 1% (Normal)')
ax.plot(dates_p, res_pt['var_forecasts'], color='green', lw=1, linestyle='--', label='VaR 1% (t-Student)')

viol_mask_pn = res_pn['violations']
ax.scatter(dates_p[viol_mask_pn], res_pn['actual_returns'][viol_mask_pn],
           color='red', s=20, zorder=5, label=f'Normal violations ({viol_mask_pn.sum()})')
ax.set_title('Case 2: Currency portfolio — VaR 1%')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Levene and Bartlett tests — VaR 1% backtesting verification
# ============================================================
# Split returns into two groups:
#   - group 1: days with VaR violations
#   - group 2: days without violations
# Test whether return variance is the same in both groups.

from scipy.stats import levene, bartlett

print("=" * 70)
print("LEVENE AND BARTLETT TESTS — VaR 1% backtesting")
print("=" * 70)

test_cases = [
    ('EUR/PLN (Normal)', bt_results_all['EUR/PLN (Normal)']),
    ('EUR/PLN (t-Student)', bt_results_all['EUR/PLN (t-Student)']),
    ('Portfolio (Normal)', bt_results_all['Portfolio (Normal)']),
    ('Portfolio (t-Student)', bt_results_all['Portfolio (t-Student)']),
]

rows = []
for name, res in test_cases:
    returns = res['actual_returns']
    viol = res['violations']
    
    group_viol = returns[viol]
    group_no_viol = returns[~viol]
    
    lev_stat, lev_p = levene(group_viol, group_no_viol)
    bart_stat, bart_p = bartlett(group_viol, group_no_viol)
    
    rows.append({
        'Model': name,
        'n violations': int(viol.sum()),
        'Levene stat': f'{lev_stat:.4f}',
        'Levene p-value': f'{lev_p:.4f}',
        'Levene result': '✗ Reject H₀' if lev_p < 0.05 else '✓ No grounds to reject.',
        'Bartlett stat': f'{bart_stat:.4f}',
        'Bartlett p-value': f'{bart_p:.4f}',
        'Bartlett result': '✗ Reject H₀' if bart_p < 0.05 else '✓ No grounds to reject.',
    })

df_tests = pd.DataFrame(rows).set_index('Model')
display(df_tests)

print("\nH₀: return variances on VaR violation days and non-violation days are equal")
print("H₁: variances differ significantly (α = 0.05)")
print("\nLevene test — robust to non-normality (uses median)")
print("Bartlett test — assumes normality of distributions in groups")


---
# Summary

## Key conclusions

1. **The t-Student distribution models financial data better** than the Normal distribution — lower KS statistic, better tail fit, which is crucial for VaR estimation.

2. **XTB currency diversification is limited** due to high correlation of exchange rates against PLN (common denominator). Nevertheless, the currency portfolio has measurably lower risk than individual components.

3. **Extreme risk (EVT)** indicates that standard models (Normal, t-Student) may underestimate risk in extreme scenarios. GEV/GPD models provide more conservative but realistic VaR estimates in the far tail.

4. **Backtesting** confirms that the t-Student model generates fewer false VaR violations than the Normal model, making it the preferred choice for risk management at XTB S.A.

5. **Practical implications for XTB:** The company should use fat-tailed models (t-Student or EVT) to determine capital requirements and exposure limits, especially in the context of currency risk and extreme stock market movements.

# 10.04
- backtesting volatility: Levene and Bartlett tests (checking volatility over time)
- heteroskedasticity test
- alpha-stable distribution for xtb.wa
